# Predict Customer Churn - Advanced XGBoost Multi-Seed Ensemble

This notebook represents our most advanced model architecture for the **"Predict Customer Churn"** Kaggle competition. It utilizes **XGBoost** with native categorical and GPU support, pushing for the highest Public LB score through robust variance reduction techniques.

**Architecture Overview:**
1.  **Rigorous Tuning:** Performs 100 Trials using Optuna. Crucially, the objective function evaluates internally across a 5-Fold Stratified CV (`xgb.cv`), identifying hyperparameters that generalize reliably rather than overfitting a single validation split.
2.  **Multi-Seed Ensembling:** After the optimal parameters are found, the notebook trains **50 distinct XGBoost models** (10 Folds × 5 explicit random seeds) and averages their test set predictions to form a highly resilient final submission.

**Hardware Switch:** Designed for seamless transition. Make sure **"GPU"** is selected in your Kaggle notebook settings.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
import warnings
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Reduce output noise

# --- 1. SETTINGS & PATHS ---
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle environment.")
    DATA_DIR = '/kaggle/input/playground-series-s6e3/'
else:
    print("Running locally.")
    DATA_DIR = '../data/'

# Configuration parameters
RUN_TUNING = True   # Set to True to run the 100 Trial sweep
N_TRIALS = 100

# Multi-Seed Ensemble settings
SEEDS = [42, 2024, 777, 1337, 999]
N_SPLITS = 10 
TOTAL_MODELS = len(SEEDS) * N_SPLITS

## 2. Load & Preprocess Data
To prevent local overfitting, we intentionally skip overly complex additive math features here and stick strictly to raw data plus label encoded categories.

In [ ]:
print(f"Loading data from {DATA_DIR}...")
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

# Target variable formatting
target = 'Churn'
if train_df[target].dtype == 'object':
    train_df[target] = train_df[target].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# Combine for unified preprocessing
train_df['is_train'] = 1
test_df['is_train'] = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', target, 'is_train']]
categorical_features = []

print("Encoding Categorials...")
for col in features:
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

# Fill missing numerical features
num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Split back
train_encoded = df[df['is_train'] == 1].reset_index(drop=True)
test_encoded = df[df['is_train'] == 0].reset_index(drop=True)

X = train_encoded[features]
y = train_encoded[target]
X_test = test_encoded[features]

# Very Important: Cast encoded categorical indices to pandas 'category' type
# XGBoost automatically leverages this for optimal splits (enable_categorical=True)
for col in categorical_features:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"Final Train shape: {X.shape}, Test shape: {X_test.shape}")

## 3. High-Fidelity Optuna Hyperparameter Tuning
In highly competitive targets, evaluating a single 80/20 train_test_split during Optuna is sub-optimal. Here, our objective function computes an extensive 5-Fold cross validation *per trial* to ensure parameters are deeply robust against variance.

In [ ]:
# Preallocate standard base parameters
best_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist', 
    'max_bin': 256,
    'verbosity': 0,
}

if IS_KAGGLE:
    best_params['device'] = 'cuda' # Kaggle GPU mapping

# Shared XGBoost evaluation matrix
dtrain = xgb.DMatrix(X, label=y, enable_categorical=True)

if RUN_TUNING:
    print(f"\nStarting {N_TRIALS}-Trial Exhaustive Optuna Tuning...")
    print("Using an internal 5-Fold 'xgb.cv' for generalization checks per trial.")
    
    def objective(trial):
        params = best_params.copy()
        params.update({
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.3, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True),
        })

        # The secret sauce: Run CV inside Optuna for robust tuning
        cv_result = xgb.cv(
            params,
            dtrain,
            num_boost_round=1500,
            nfold=5,
            stratified=True,
            metrics='auc',
            early_stopping_rounds=50,
            verbose_eval=False,
            seed=42
        )
        
        # Output the peak AUC across the valid folds
        return cv_result['test-auc-mean'].max()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    print(f"\n[OPTUNA COMPLETE] Best Internal 5-Fold CV AUC: {study.best_value:.5f}")
    print("Parameters derived:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")
        best_params[k] = v
        
else:
    print("\nSkipping Optuna. Loading known generalized defaults...")
    precomputed_params = {
        'learning_rate': 0.0125,
        'max_depth': 6,
        'min_child_weight': 10,
        'subsample': 0.825,
        'colsample_bytree': 0.612,
        'colsample_bylevel': 0.852,
        'gamma': 0.125,
        'reg_alpha': 0.0452,
        'reg_lambda': 2.341,
    }
    best_params.update(precomputed_params)

## 4. Multi-Seed Ensemble (Variant Variance Reduction)
To achieve a premier stable score on the public LB, we train 50 individual XGBoost models traversing differing dataset splitting bounds triggered by explicit global seeds. We then average the output.

In [ ]:
print(f"\nInitiating Multi-Seed Massive Ensemble")
print(f"Architecture: {len(SEEDS)} Seeds × {N_SPLITS}-Folds = {TOTAL_MODELS} Total XGBoost Models")

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))

dtest = xgb.DMatrix(X_test, enable_categorical=True)

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n===> Seed Sequence {seed} ({seed_idx + 1}/{len(SEEDS)}) <===")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    seed_oof = np.zeros(len(X))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain_fold = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dval_fold = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
        
        model_fold = xgb.train(
            best_params,
            dtrain_fold,
            num_boost_round=3000, 
            evals=[(dval_fold, 'validation')],
            early_stopping_rounds=150,
            verbose_eval=False
        )
        
        # Predict local validation
        fold_val_preds = model_fold.predict(dval_fold, iteration_range=(0, model_fold.best_iteration + 1))
        seed_oof[val_idx] = fold_val_preds
        global_oof_sum[val_idx] += fold_val_preds
        
        # Predict test data
        fold_test_preds = model_fold.predict(dtest, iteration_range=(0, model_fold.best_iteration + 1))
        test_preds_sum += fold_test_preds
        
        print(f"   Fold {fold+1:02d} | Valid AUC: {roc_auc_score(y_val, fold_val_preds):.5f}")
        
    # Review specific seed aggregate stability
    seed_auc = roc_auc_score(y, seed_oof)
    print(f"   >>> Internal OOF AUC for Seed {seed}: {seed_auc:.5f}")

# Calculate final blended validation metric
global_oof_avg = global_oof_sum / len(SEEDS)
final_oof_auc = roc_auc_score(y, global_oof_avg)

print("\n======================================================")
print(f"FINAL AVERAGED MULTI-SEED ENSEMBLE OOF AUC: {final_oof_auc:.5f}")
print("======================================================")

## 5. Generate Submission File

In [ ]:
print(f"\nBlending test outputs across all {TOTAL_MODELS} trained iterations...")
final_test_preds = test_preds_sum / TOTAL_MODELS

sub[target] = final_test_preds
sub.to_csv('submission_xgboost_multiseed_100t.csv', index=False)

print("Submission generated: 'submission_xgboost_multiseed_100t.csv'")
display(sub.head())